# Extrinsic Coarse Symmetry Scoring

This notebook is a small synthetic sanity check for `organograph.mesh.symmetry`. The scores are not hard labels. Lower trimmed RMS means a better approximate symmetry, while higher matched fraction means more of the sampled surface lies within the relative tolerance after applying that symmetry.

In [21]:
import numpy as np

from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.symmetry import (
    best_overall_symmetry,
    best_symmetry_per_level,
    characteristic_size,
    describe_symmetry_score,
    laplace_beltrami_low_pass_vertices,
    pca_candidate_axes,
    reflection_matrix,
    rotation_matrix,
    run_multiscale_symmetry_analysis,
    sample_surface_points,
    score_transformation,
    surface_area_centroid,
    symmetry_results_to_records,
)
from organograph.plotting.meshes import plot_organoid_mesh

## Run on One Organoid Mesh

Replace `MESH_PATH` with an STL, OBJ, or VTP path. The pipeline loads the mesh, reconstructs low-pass coordinates through Laplace-Beltrami level `l` using `l**2` modes, samples the smoothed surface by area, and reports the best reflection, C2, and C3 candidates at each smoothing level.

In [22]:
# MESH_PATH = "path/to/organoid_mesh.vtp"
# mesh = OrganoidMesh(MESH_PATH)
# results = run_multiscale_symmetry_analysis(
#     mesh,
#     l_values=[3, 5, 8, 10],
#     n_samples=8000,
#     trim_fraction=0.95,
#     close_threshold=0.05,
#     random_seed=0,
# )
# records = symmetry_results_to_records(results, organoid_id="sample_001")
# try:
#     import pandas as pd
#     display(pd.DataFrame(records))
# except ImportError:
#     records
# describe_symmetry_score(best_overall_symmetry(results))

## Synthetic Mesh Utilities

These simple radial meshes are not intended to be biological models. They are quick checks that the score behaves sensibly on sphere, ellipsoid, peanut, tripod, asymmetric, and noisy symmetric cases.

In [23]:
def make_uv_sphere(n_lat=18, n_lon=36, radius_fn=None):
    vertices = [[0.0, 0.0, 1.0]]
    for i in range(1, n_lat):
        theta = np.pi * i / n_lat
        for j in range(n_lon):
            phi = 2.0 * np.pi * j / n_lon
            r = 1.0 if radius_fn is None else float(radius_fn(theta, phi))
            vertices.append([
                r * np.sin(theta) * np.cos(phi),
                r * np.sin(theta) * np.sin(phi),
                r * np.cos(theta),
            ])
    vertices.append([0.0, 0.0, -1.0])
    bottom = len(vertices) - 1
    faces = []
    for j in range(n_lon):
        faces.append([0, 1 + j, 1 + (j + 1) % n_lon])
    for i in range(n_lat - 2):
        ring0 = 1 + i * n_lon
        ring1 = ring0 + n_lon
        for j in range(n_lon):
            a = ring0 + j
            b = ring0 + (j + 1) % n_lon
            c = ring1 + j
            d = ring1 + (j + 1) % n_lon
            faces.append([a, c, b])
            faces.append([b, c, d])
    last_ring = 1 + (n_lat - 2) * n_lon
    for j in range(n_lon):
        faces.append([last_ring + j, bottom, last_ring + (j + 1) % n_lon])
    return np.asarray(vertices, float), np.asarray(faces, np.int64)


def as_mesh(vertices, faces):
    return OrganoidMesh().load_from_arrays(vertices, faces)


def build_synthetic_cases(seed=2):
    rng = np.random.default_rng(seed)
    base_v, faces = make_uv_sphere()

    sphere = base_v.copy()
    ellipsoid = base_v * np.array([2.0, 1.2, 0.75])

    def peanut_radius(theta, phi):
        x = np.sin(theta) * np.cos(phi)
        return 0.85 + 0.35 * x * x

    peanut, _ = make_uv_sphere(radius_fn=peanut_radius)
    peanut[:, 0] *= 1.8

    def tripod_radius(theta, phi):
        return 1.0 + 0.35 * (np.sin(theta) ** 2) * np.cos(3.0 * phi)

    tripod, _ = make_uv_sphere(radius_fn=tripod_radius)

    asymmetric = ellipsoid.copy()
    asymmetric += 0.18 * np.column_stack([
        asymmetric[:, 1] * asymmetric[:, 2],
        asymmetric[:, 0] ** 2,
        np.sin(3 * asymmetric[:, 0]),
    ])

    noisy = ellipsoid + 0.08 * rng.normal(size=ellipsoid.shape)

    return {
        "sphere": sphere,
        "ellipsoid": ellipsoid,
        "peanut": peanut,
        "tripod": tripod,
        "asymmetric": asymmetric,
        "noisy_ellipsoid": noisy,
    }, faces

In [24]:
cases, faces = build_synthetic_cases()
mesh = as_mesh(cases["tripod"], faces)
plot_organoid_mesh(mesh, backend="plotly")

## Score the Synthetic Cases

Including `None` scores the raw mesh. The integer values score LB-smoothed meshes reconstructed through level `l`, which uses `l**2` modes via the `OrganoidMesh` spectral reconstruction API. For a noisy but coarsely symmetric shape, the raw row may look worse while the low-pass rows recover the expected coarse symmetry.

The dataframe columns mean:

- `organoid_id`: synthetic example name, or your sample ID for real data.
- `l`: Laplace-Beltrami reconstruction level; `None` means the raw mesh was scored.
- `n_modes`: number of retained modes, equal to `l**2`; `None` for the raw mesh.
- `symmetry`: candidate symmetry family being scored: `reflection`, `C2` rotation, or `C3` rotation.
- `best_axis`: PCA candidate axis that gave the best score for that symmetry family at this smoothing level.
- `candidate_source`: where the candidate axis came from. Currently this is `PCA`; LB-derived axes are reserved for later validation.
- `axis_x`, `axis_y`, `axis_z`: coordinates of the selected candidate axis or mirror-plane normal after centering.
- `trimmed_rms`: robust normalized mismatch after discarding the largest-distance tail; lower means stronger approximate coarse symmetry.
- `normalized_rms`: RMS mismatch normalized by organoid size; lower is better but more sensitive to outliers than `trimmed_rms`.
- `median`: median normalized mismatch; describes the typical sampled surface-point error.
- `matched_fraction`: fraction of sampled surface points within the relative tolerance after applying the transform; higher means more of the surface is explained by the symmetry.
- `n_samples`: number of area-weighted surface samples used for scoring.
- `characteristic_scale`: RMS surface radius used to make distances dimensionless.
- `close_threshold`: relative distance cutoff used to compute `matched_fraction`.

In [25]:
all_records = []
results_by_name = {}
meshes_by_name = {}
l_values = [None, 3, 5, 8]
max_l = max(l for l in l_values if l is not None)

for name, vertices in cases.items():
    mesh = as_mesh(vertices, faces)
    mesh.compute_spectral_coefficients(lmax=max_l)  # computes max_l**2 modes once; lower l levels reuse them
    results = run_multiscale_symmetry_analysis(
        mesh,
        l_values=l_values,
        n_samples=5000,
        close_threshold=0.05,
        random_seed=3,
    )
    meshes_by_name[name] = mesh
    results_by_name[name] = results
    all_records.extend(symmetry_results_to_records(results, organoid_id=name))
    print(name, "->", describe_symmetry_score(best_overall_symmetry(results)))

try:
    import pandas as pd
    df = pd.DataFrame(all_records)
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", 0,
        "display.max_colwidth", None,
    ):
        display(df)
except ImportError:
    for row in all_records:
        print(row)

[Info] Eigen-decomposition not found. Computing now...
sphere -> Best coarse symmetry: C2 at l=8, trimmed RMS=0.025, matched area fraction=0.96.
[Info] Eigen-decomposition not found. Computing now...
ellipsoid -> Best coarse symmetry: C2 at raw mesh, trimmed RMS=0.025, matched area fraction=0.97.
[Info] Eigen-decomposition not found. Computing now...
peanut -> Best coarse symmetry: reflection at l=8, trimmed RMS=0.024, matched area fraction=0.97.
[Info] Eigen-decomposition not found. Computing now...
tripod -> Best coarse symmetry: C3 at l=8, trimmed RMS=0.028, matched area fraction=0.94.
[Info] Eigen-decomposition not found. Computing now...
asymmetric -> Best coarse symmetry: C2 at raw mesh, trimmed RMS=0.025, matched area fraction=0.96.
[Info] Eigen-decomposition not found. Computing now...
noisy_ellipsoid -> Best coarse symmetry: C2 at l=5, trimmed RMS=0.030, matched area fraction=0.93.


,organoid_id,l,n_modes,symmetry,best_axis,candidate_source,axis_x,axis_y,axis_z,trimmed_rms,normalized_rms,median,matched_fraction,n_samples,characteristic_scale,close_threshold
0,sphere,NaN,NaN,reflection,PCA1,PCA,0.900644,-0.200694,-0.385437,0.025764,0.028192,0.023222,0.9498,5000,0.995778,0.05
1,sphere,NaN,NaN,C2,PCA2,PCA,0.354672,0.851986,0.385133,0.025557,0.027919,0.022909,0.9588,5000,0.995778,0.05
2,sphere,NaN,NaN,C3,PCA3,PCA,0.251093,-0.483572,0.838517,0.026047,0.028389,0.023577,0.9535,5000,0.995778,0.05
3,sphere,3.0,9.0,reflection,PCA2,PCA,0.198142,-0.676362,0.709418,0.025925,0.028356,0.023372,0.9524,5000,0.995766,0.05
4,sphere,3.0,9.0,C2,PCA3,PCA,-0.052314,0.715439,0.696714,0.025813,0.028171,0.023550,0.9560,5000,0.995766,0.05
5,sphere,3.0,9.0,C3,PCA1,PCA,0.978776,0.175161,-0.106375,0.026189,0.028603,0.023647,0.9490,5000,0.995766,0.05
6,sphere,5.0,25.0,reflection,PCA2,PCA,0.735164,-0.436403,0.518735,0.025607,0.028056,0.022753,0.9534,5000,0.995773,0.05
7,sphere,5.0,25.0,C2,PCA2,PCA,0.735164,-0.436403,0.518735,0.026289,0.028717,0.023795,0.9492,5000,0.995773,0.05
8,sphere,5.0,25.0,C3,PCA2,PCA,0.735164,-0.436403,0.518735,0.025931,0.028316,0.023431,0.9551,5000,0.995773,0.05
9,sphere,8.0,64.0,reflection,PCA3,PCA,-0.485737,-0.552253,0.677552,0.025593,0.027857,0.023524,0.9634,5000,0.995775,0.05


## Visualize All Tested Symmetry Transforms

This cell chooses one organoid and one smoothing level, then displays every PCA-axis candidate tested by the scoring pipeline. Columns are symmetry groups (`reflection`, `C2`, `C3`) and rows are candidate axes (`PCA1`, `PCA2`, `PCA3`). Each panel overlays the original low-pass surface in gray with the transformed surface for that specific candidate. The figure uses Plotly, so each panel can be rotated and zoomed interactively. For C3, the displayed transform is the 120-degree rotation; the score shown combines both 120 and 240 degrees.

In [ ]:
from scipy.spatial import cKDTree
from plotly.subplots import make_subplots
import plotly.graph_objects as go

display_organoid = "asymmetric"
display_l = None
mesh = meshes_by_name[display_organoid]
level_label = "raw mesh" if display_l is None else f"l={display_l}"

vertices = laplace_beltrami_low_pass_vertices(mesh, display_l)
faces = np.asarray(mesh.f, dtype=np.int64)
centroid = surface_area_centroid(vertices, faces)
centered_vertices = vertices - centroid[None, :]

viz_points = sample_surface_points(centered_vertices, faces, n_samples=5000, rng=17)
target_tree = cKDTree(viz_points)
scale = characteristic_size(viz_points)
candidate_axes = pca_candidate_axes(viz_points)
symmetry_groups = ["reflection", "C2", "C3"]

def matrices_for_symmetry(symmetry, axis):
    if symmetry == "reflection":
        return [reflection_matrix(axis)]
    if symmetry == "C2":
        return [rotation_matrix(axis, np.pi)]
    if symmetry == "C3":
        return [
            rotation_matrix(axis, 2.0 * np.pi / 3.0),
            rotation_matrix(axis, 4.0 * np.pi / 3.0),
        ]
    raise ValueError(symmetry)

def combined_candidate_score(matrices):
    scores = [
        score_transformation(
            viz_points,
            target_tree,
            matrix,
            scale,
            trim_fraction=0.95,
            close_threshold=0.05,
        )
        for matrix in matrices
    ]
    return {key: float(np.mean([score[key] for score in scores])) for key in scores[0]}

candidate_grid = []
subplot_titles = []
for axis_name, axis, _source in candidate_axes:
    row_entries = []
    for symmetry in symmetry_groups:
        matrices = matrices_for_symmetry(symmetry, axis)
        metrics = combined_candidate_score(matrices)
        row_entries.append((axis_name, axis, symmetry, matrices[0], metrics))
        subplot_titles.append(
            f"{axis_name} | {symmetry}<br>"
            f"trimmed RMS={metrics['trimmed_rms']:.3f}, "
            f"matched={metrics['matched_fraction']:.2f}"
        )
    candidate_grid.append(row_entries)

axis_limit = float(np.max(np.abs(centered_vertices)) * 1.08)
colors = {
    "reflection": "rgba(31, 119, 180, 0.45)",
    "C2": "rgba(255, 127, 14, 0.45)",
    "C3": "rgba(44, 160, 44, 0.45)",
}

fig = make_subplots(
    rows=len(candidate_axes),
    cols=len(symmetry_groups),
    specs=[[{"type": "scene"} for _ in symmetry_groups] for _ in candidate_axes],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.01,
    vertical_spacing=0.04,
)

for row, row_entries in enumerate(candidate_grid, start=1):
    for col, (axis_name, axis, symmetry, matrix, metrics) in enumerate(row_entries, start=1):
        transformed_vertices = centered_vertices @ matrix.T
        axis = axis / np.linalg.norm(axis)
        axis_line = np.vstack([-axis_limit * axis, axis_limit * axis])
        show_legend = row == 1 and col == 1

        fig.add_trace(
            go.Mesh3d(
                x=centered_vertices[:, 0],
                y=centered_vertices[:, 1],
                z=centered_vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color="lightgray",
                opacity=0.24,
                name="original low-pass mesh",
                showlegend=show_legend,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Mesh3d(
                x=transformed_vertices[:, 0],
                y=transformed_vertices[:, 1],
                z=transformed_vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color=colors[symmetry],
                opacity=0.45,
                name="transformed candidate",
                showlegend=show_legend,
                hovertemplate=(
                    f"{axis_name} | {symmetry}<br>"
                    f"trimmed RMS={metrics['trimmed_rms']:.3f}<br>"
                    f"matched={metrics['matched_fraction']:.2f}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter3d(
                x=axis_line[:, 0],
                y=axis_line[:, 1],
                z=axis_line[:, 2],
                mode="lines",
                line=dict(color="black", width=5),
                name="candidate axis / plane normal",
                showlegend=show_legend,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )

scene_settings = dict(
    xaxis=dict(range=[-axis_limit, axis_limit], visible=False),
    yaxis=dict(range=[-axis_limit, axis_limit], visible=False),
    zaxis=dict(range=[-axis_limit, axis_limit], visible=False),
    aspectmode="cube",
    bgcolor="rgba(0,0,0,0)",
)
for row in range(1, len(candidate_axes) + 1):
    for col in range(1, len(symmetry_groups) + 1):
        fig.update_scenes(scene_settings, row=row, col=col)

fig.update_layout(
    title=f"{display_organoid}: all tested symmetry candidates on {level_label}",
    width=1320,
    height=1080,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    margin=dict(l=0, r=0, b=0, t=110),
)
fig.show()

## Interpretation Notes

- A sphere should score well for many candidate rotations and reflections, so the best class is not biologically unique.
- A triaxial ellipsoid should show strong reflection symmetry and often strong C2 rotations around PCA axes.
- A peanut/dumbbell should usually score well for C2 around the long PCA axis, with reflections also plausible.
- A tripod should prefer C3 around the axis perpendicular to the three lobes.
- An asymmetric perturbation should have larger trimmed RMS and lower matched fraction.
- If noisy raw rows improve after low-pass reconstruction, that indicates coarse symmetry with fine-scale/noise asymmetry rather than exact full-surface symmetry.